# Early fusion

## Early fusion with Random Forest Model

In [1]:
import pandas as pd
import numpy as np
import warnings
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, cross_validate, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.exceptions import ConvergenceWarning

In [2]:
df = pd.read_pickle(r"C:\Users\Juli\Documents\Master\Projekt Genomforschung\Datasets\harmonized_data.pkl")

In [3]:
def early_fusion(df, target, model_name):
    if 'Bit_0' not in df.columns:
        fp_df = pd.DataFrame(df['MorganFP'].tolist(), index=df.index).astype('uint8')
        fp_df.columns = [f'Bit_{i}' for i in range(fp_df.shape[1])]
        df = pd.concat([df, fp_df], axis=1)
        
    if 'Donor' not in df.columns:
        expanded_features = pd.DataFrame(df['PharmacophoreFeatures'].tolist(), 
                                         index=df.index).astype('float32')
        df = pd.concat([df.drop('PharmacophoreFeatures', axis=1), expanded_features], axis=1)
        df = df.fillna(0)
    # training set for genomic features
    X = pd.concat([df.loc[:, ['Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder']],
                        df.loc[:, df.columns.str.startswith('Bit_')],
                        df.filter(regex=r'.* \(.*\)')], axis=1)
    X_asList = X.values.astype('float32')
    y = df[target].values.astype('float32') # y: drug response
    # train-test split
    gss = GroupShuffleSplit(test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(X, y, groups=df['ModelID'].values))
    X_train, X_test = X_asList[train_idx], X_asList[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    groups = df.iloc[train_idx]['ModelID'].values # to ensure held-out validation

    if model_name == 'RandomForest':
        # initialize the Random Forest Regressor
        model = RandomForestRegressor(
            n_estimators=100,
            max_depth=15,
            min_samples_leaf=5,
            max_features='sqrt',
            n_jobs=-1,
            random_state=42
        )

        # perform Cross-Validation
        print("Starting Cross-Validation on Training Data...")
        cv_results = cross_validate(
            model, X_train, y_train, 
            groups=groups, 
            cv=GroupKFold(n_splits=5),
            scoring=['neg_mean_squared_error', 'r2'],
            return_train_score=True
        )

        # output Results
        mse_scores = -cv_results['test_neg_mean_squared_error']
        rmse_scores = np.sqrt(mse_scores)
        r2_scores = cv_results['test_r2']

        print(f"--- Multimodal (Random Forest - Early Fusion) Performance ---")
        print(f"R² Score: {np.mean(r2_scores):.4f}")
        print(f"RMSE:     {np.mean(rmse_scores):.4f}")
        print(f"------------------------------------")

        # fit the model on the training data
        model.fit(X_train, y_train)

        # test fit
        y_pred = model.predict(X_test)
        test_r2 = r2_score(y_test, y_pred)
        test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        print(f"Test R² Score: {test_r2:.4f}")
        print(f"Test RMSE:     {test_rmse:.4f}")
        # feature importance
        all_features = X.columns
        importances = model.feature_importances_
        gene_cols = X.filter(regex=r'.* \(.*\)').columns
        features_df = pd.DataFrame({
            'Feature_Name': all_features,
            'Importance': importances,
            'Type': ['Genetic' if f in gene_cols else 'Molecular structure' for f in all_features]
        })
        # Filtere irrelevante Features heraus (Importance nahe 0)
        selected_features = features_df[features_df['Importance'] > 0.0001].copy()
        chem_selected = selected_features[selected_features['Type'] == 'Molecular structure']
        bio_selected = selected_features[selected_features['Type'] == 'Genetic']
        
        print(f"Total number of selected features: {len(selected_features)} von {len(all_features)}")
        print(f" -> Used chemical features:  {len(chem_selected)}")
        print(f" -> Used biological genes:    {len(bio_selected)}")
        
        print(f"\nTop 10 most important features:")
        print(features_df.sort_values(by='Importance', ascending=False).head(10))

    elif model_name == 'ElasticNet':
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('model', ElasticNetCV(
                l1_ratio=[0.001, 0.01, 0.05, 0.1, 0.5, 0.7, 0.9, 0.99, 1], # model will find the best mix
                cv=GroupKFold(n_splits=5).split(X_train, y_train, groups=groups),
                random_state=42,
                max_iter=8000,
                alphas=20,
                tol=1e-3,
                n_jobs=-1
            ))
        ])

        pipeline.fit(X_train, y_train)

        train_r2_global = pipeline.score(X_train, y_train)
        fitted_model = pipeline.named_steps['model']
        best_alpha_idx = np.where(fitted_model.alphas_ == fitted_model.alpha_)[0][0]
        mean_mse_best_alpha = np.mean(fitted_model.mse_path_[best_alpha_idx])
        variance_y_train = np.var(y_train)
        train_cv_r2 = 1 - (mean_mse_best_alpha / variance_y_train)

        y_pred = pipeline.predict(X_test)
        test_r2 = pipeline.score(X_test, y_test)
        test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))

        print("\n" + "="*40)
        print(f"--- Multimodal (Elastic Net - Early Fusion) Performance ---")
        print("="*40)
        print(f"Chosen Alpha:           {fitted_model.alpha_:.6f}")
        print(f"Chosen L1-Ratio:        {fitted_model.l1_ratio_:.2f}")
        print("-"*40)
        print(f"Global Training R² Score:  {train_r2_global:.4f}")
        print(f"Internal CV Trainings R²:     {train_cv_r2:.4f}")
        print(f"Held-Out Test R² Score:       {test_r2:.4f}")
        print("="*40)

        print(f"Test RMSE:     {test_rmse:.4f}")

        # analyze feature importance
        final_model = pipeline.named_steps['model']
        coefs = final_model.coef_

        # Create a summary table
        feature_names = X.columns
        features_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': coefs})
        features_df['Abs_Coef'] = features_df['Coefficient'].abs()

        # Filter for features the model didn't set to zero
        selected_features = features_df[features_df['Coefficient'] != 0]

        print(f"\nElastic Net selected {len(selected_features)} features out of {len(feature_names)}.")
        print(f"Top 5 Positive Features (Increase {target}):")
        print(features_df.sort_values(by='Coefficient', ascending=False).head(5))
        print(f"\nTop 5 Negative Features (Decrease {target}):")
        print(features_df.sort_values(by='Coefficient', ascending=True).head(5))
            
    

In [ ]:
early_fusion(df, target='LN_IC50', model_name='RandomForest')

Starting Cross-Validation on Training Data...
--- Multimodal (early fusion) Performance ---
R² Score: 0.5544
RMSE:     1.8868
------------------------------------
Test R² Score: 0.5546
Test RMSE:     1.8753
Total number of selected features: 921 von 2010
 -> Used chemical features:  651
 -> Used biological genes:    270

Top 10 most important features:
     Feature_Name  Importance                 Type
455       Bit_447    0.013441  Molecular structure
0           Donor    0.013330  Molecular structure
3      Hydrophobe    0.012498  Molecular structure
584       Bit_576    0.011212  Molecular structure
719       Bit_711    0.010133  Molecular structure
522       Bit_514    0.009569  Molecular structure
895       Bit_887    0.009402  Molecular structure
6    NegIonizable    0.008754  Molecular structure
641       Bit_633    0.008677  Molecular structure
432       Bit_424    0.008551  Molecular structure


In [ ]:
early_fusiont(df, target='AUC', model_name='RandomForest')

Starting Cross-Validation on Training Data...
--- Multimodal (early fusion) Performance ---
R² Score: 0.4910
RMSE:     0.1067
------------------------------------
Test R² Score: 0.5104
Test RMSE:     0.1013
Total number of selected features: 1438 von 2010
 -> Used chemical features:  630
 -> Used biological genes:    808

Top 10 most important features:
    Feature_Name  Importance                 Type
522      Bit_514    0.019529  Molecular structure
584      Bit_576    0.015900  Molecular structure
573      Bit_565    0.014119  Molecular structure
484      Bit_476    0.012637  Molecular structure
577      Bit_569    0.012171  Molecular structure
719      Bit_711    0.011588  Molecular structure
367      Bit_359    0.011096  Molecular structure
410      Bit_402    0.010125  Molecular structure
7       ZnBinder    0.009546  Molecular structure
609      Bit_601    0.009128  Molecular structure


In [ ]:
early_fusion(df, target='LN_IC50', model_name='ElasticNet')